# PUMAS Data Flow: From CSV to Dashboard

This notebook shows step-by-step how data flows from CSV files to the dashboard.

## Step 1: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

## Step 2: Load CSV Data Files

In [3]:
# Load 4 CSV files from data/processed/
gps_df = pd.read_csv('../data/processed/gps_trips.csv')
traffic_df = pd.read_csv('../data/processed/traffic_flow.csv')
zone_df = pd.read_csv('../data/processed/zone_travel_times.csv')
weather_df = pd.read_csv('../data/processed/weather_data.csv')

print("=== LOADED DATA ===")
print(f"GPS Trips: {len(gps_df)} rows")
print(f"Traffic Flow: {len(traffic_df)} rows")
print(f"Zone Travel Times: {len(zone_df)} rows")
print(f"Weather Data: {len(weather_df)} rows")

=== LOADED DATA ===
GPS Trips: 5000 rows
Traffic Flow: 240 rows
Zone Travel Times: 90 rows
Weather Data: 720 rows


## Step 3: Initialize DataPipeline

In [4]:
from src.data.data_pipeline import DataPipeline

pipeline = DataPipeline()
print("DataPipeline initialized successfully!")

DataPipeline initialized successfully!


## Step 4: Get Nairobi Zones (Geographic Data)

In [5]:
from src.data.data_pipeline import get_nairobi_zones

zones = get_nairobi_zones()
print("=== NAIROBI ZONES ===")
for name, info in zones.items():
    print(f"{name}: lat={info['lat']}, lon={info['lon']}, risk={info['risk_level']}")

=== NAIROBI ZONES ===
CBD: lat=-1.286, lon=36.8178, risk=high
Westlands: lat=-1.2644, lon=36.8032, risk=medium
Kilimani: lat=-1.2956, lon=36.819, risk=medium
Kasarani: lat=-1.2172, lon=36.898, risk=medium
Karen: lat=-1.3186, lon=36.678, risk=low
Embakasi: lat=-1.3274, lon=36.9273, risk=high
Ruiru: lat=-1.1467, lon=36.9581, risk=low
Machakos: lat=-1.5177, lon=37.2634, risk=medium
Nairobi West: lat=-1.3087, lon=36.837, risk=high
Parklands: lat=-1.2711, lon=36.8402, risk=medium


## Step 5: Trip Planning - Compare Modes

In [6]:
# Compare transport modes from CBD to Karen
result = pipeline.compare_modes('CBD', 'Karen')
print("=== TRIP: CBD → Karen ===")
print(f"Distance: {result['distance_km']} km")
print(f"Walking: {result['modes']['walking']['time']} (Cost: {result['modes']['walking']['cost']} KES)")
print(f"Matatu: {result['modes']['matatu']['time']} (Cost: {result['modes']['matatu']['cost']} KES)")
print(f"Driving: {result['modes']['driving']['time']} (Cost: {result['modes']['driving']['cost']} KES)")

=== TRIP: CBD → Karen ===
Distance: 15.9 km
Walking: 3 hr 11 min (Cost: 0 KES)
Matatu: 47 min 47 sec (Cost: 30 KES)
Driving: 31 min 51 sec (Cost: 243 KES)


## Step 6: Descriptive Analytics - What Happened?

In [8]:
# 7-day traffic trends
trends = pipeline.get_7day_trends()
print("=== 7-DAY TRAFFIC TRENDS ===")
print(f"Dates: {trends['dates']}")
print(f"Traffic Index: {[round(x,2) for x in trends['traffic_index']]}")

# Statistics summary - FIXED
stats = pipeline.get_statistics_summary()
traffic_stats = stats.get('traffic', {})
if 'traffic_index' in traffic_stats:
    print(f"\nAvg Traffic Index: {traffic_stats['traffic_index']['avg']:.2f}")
else:
    print(f"\nTraffic stats keys: {traffic_stats.keys()}")
print(f"Total Trips: {stats['trips']['total_trips']}")


=== 7-DAY TRAFFIC TRENDS ===
Dates: ['2026-03-27', '2026-03-28', '2026-03-29', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02']
Traffic Index: [0.85, 0.82, 1.26, 1.18, 1.08, 1.27, 1.21]

Traffic stats keys: dict_keys(['avg', 'min', 'max', 'std'])
Total Trips: 0


## Step 7: Diagnostic Analytics - Why?

In [9]:
# Traffic cause breakdown
cause = pipeline.get_traffic_cause_breakdown()
print("=== TRAFFIC CAUSE BREAKDOWN ===")
print(f"Time factor: {cause['time_factor']}%")
print(f"Weather factor: {cause['weather_factor']}%")
print(f"Location factor: {cause['location_factor']}%")
print(f"Explanation: {cause['explanation']}")

=== TRAFFIC CAUSE BREAKDOWN ===
Time factor: 78.9%
Weather factor: 5.3%
Location factor: 15.8%
Explanation: Morning rush hour (78%)


## Step 8: Predictive Analytics - What Will Happen?

In [10]:
# 24-hour traffic prediction
forecast = pipeline.predict_24h_traffic()
print("=== 24-HOUR TRAFFIC FORECAST ===")
print(f"Peak hours: {[p['hour'] for p in forecast['predictions'] if p['predicted_index'] > 1.5]}")

# Congestion warnings
warnings = pipeline.get_congestion_warnings()
print(f"\nActive warnings: {len(warnings['warnings'])}")
for w in warnings['warnings']:
    print(f"  - {w['zone']}: {w['message']}")

=== 24-HOUR TRAFFIC FORECAST ===
Peak hours: [7, 8, 9, 17, 18, 19, 20]

Active warnings: 1
  - CBD: Heavy congestion expected due to morning rush hour


## Step 9: Weather Integration

In [11]:
from src.data.weather_api import OpenWeatherMapAPI

weather = OpenWeatherMapAPI().get_current_weather("Nairobi")
print("=== CURRENT NAIROBI WEATHER ===")
print(f"Condition: {weather['weather_description']}")
print(f"Temperature: {weather['temperature_c']}°C")
print(f"Humidity: {weather['humidity_percent']}%")

=== CURRENT NAIROBI WEATHER ===
Condition: broken clouds
Temperature: 19.9°C
Humidity: 73%


## Step 10: Dashboard Integration

The data flows to Streamlit dashboard via `src/dashboard/app.py`

To run the dashboard:
```powershell
python -m streamlit run src/dashboard/app.py --server.port 8510
```

Then open http://localhost:8510